# Simple RAG

**Part 1: Prepare, Split and Indext Knowledge for Storing in Vector Databases**

1. Start with data: download and prepare the data you want to add as knowledge. We will extract data from some blog posts found at Lil's Blog (<https://lilianweng.github.io>) into LllamaIndex Documents.
2. Split the Documents into Chunks.
3. Compute Embedding Vectors and store them in Vector Database

## 1. Download and prepare the data

In [1]:
# Use Beautiful Soup for Web-Crawling: https://www.crummy.com/software/BeautifulSoup/
# Load blog posts from "https://lilianweng.github.io/posts/"
import bs4
from urllib.request import urlopen
from bs4 import BeautifulSoup as soup

url = "https://lilianweng.github.io/posts/"

# opening connection, grabbing the HTML from the page
with urlopen(url) as client:
    html = client.read()

soup = soup(html, 'html.parser')

In [2]:
#<a aria-label=".." class="entry-link" href="https://lilianweng.github.io/posts/2024-07-07-hallucination/"></a>
blog_posts = []
cells = soup.find_all("a", attrs={"class": "entry-link"})
for cell in cells:
    if type(cell) == bs4.element.Tag:
        blog_posts.append( {'label': cell.get('aria-label'), 'link': cell.get('href')} )
print(f"{len(blog_posts)} posts found.")   

20 posts found.


In [3]:
from llama_index.readers.web import BeautifulSoupWebReader
docs = []
for blog_post in blog_posts:
    url = [blog_post["link"]]
    docs.extend(BeautifulSoupWebReader().load_data(url))
print(len(docs))

20


In [4]:
# Now we have a list of LlamaIndex Documents. A Document is an object with some content (str) and metadata (dict).
print(docs[3].metadata)
print("Page content:", type(docs[0].get_content()))

{'URL': 'https://lilianweng.github.io/posts/2024-04-12-diffusion-video/'}
Page content: <class 'str'>


## 2. Split the Documents

Now we split each Document into chunks for embedding and vector storage. This should help us retrieve only the most relevant parts of the blog post at run time.

We split our documents into chunks of 512 Tokens per Chunk with an overlap between chunks. 

The overlap helps mitigate the possibility of separating a statement from important context related to it.

In [5]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings
# We use sentence-transformers/all-MiniLM-L6-v2 provided by HuggingFace
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Chunking-Strategie
Settings.node_parser = SentenceSplitter(
    chunk_size=512,      # Tokens pro Chunk
    chunk_overlap=50     # Überlappung
)

## 3. Initialize the Vector Store

In [6]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
import chromadb
chroma_client = chromadb.PersistentClient(
    settings=chromadb.Settings(
        persist_directory="./chroma_db"
    )
)

chroma_collection = chroma_client.get_or_create_collection(
    name="lilianweng"
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

## 4. Compute Embedding Vectors and store them in Vector Database

Now we need to index our text chunks so that we can search over them at runtime. The most common way to do this is to embed the contents of each document split and insert these embeddings into a vector database (or vector store). When we want to search over our splits, we take a text search query, embed it, and perform some sort of “similarity” search to identify the stored splits with the most similar embeddings to our query embedding. The simplest similarity measure is cosine similarity — we measure the cosine of the angle between each pair of embeddings (which are high dimensional vectors).

We can embed and store all of our document splits in a single command using the Chroma vector store and HuggingFace embedding model.

In [8]:
len(docs)

20

In [7]:
#Dokuments are chunked, embedded and stored in ChromaDB (./chroma_db)
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    docs,
    storage_context=storage_context,
)

# 5. Inspect Vector Database

In [9]:
retriever = index.as_retriever(
    similarity_top_k=5
)

nodes = retriever.retrieve(
    "Is There an article by Francis Galton published in Nature?"
)

for node in nodes:
    print("Score:", node.score)
    print(node.node.text)
    print(40*"=")

Score: 0.25581870811756485
(Feb 2024). “Thinking about High-Quality Human Data”. Lil’Log. https://lilianweng.github.io/posts/2024-02-05-human-data-quality/.

Or
@article{weng2024humandata,
  title   = "Thinking about High-Quality Human Data",
  author  = "Weng, Lilian",
  journal = "lilianweng.github.io",
  year    = "2024",
  month   = "Feb",
  url     = "https://lilianweng.github.io/posts/2024-02-05-human-data-quality/"
}
References#
[1] Francis Galton “Vox populi”  Nature 75, 450-451 (1907).
[2] Sambasivan et al. “Everyone wants to do the model work, not the data work”: Data Cascades in High-Stakes AI" CHI 2021
[3] Chris Callison-Burch. “Fast, Cheap, and Creative: Evaluating Translation Quality Using Amazon’s Mechanical Turk” EMNLP 2009
[4] Rottger et al. “Two Contrasting Data Annotation Paradigms for Subjective NLP Tasks” NAACL 2022.
[5] Aroyo & Welty “Truth Is a Lie: Crowd Truth and the Seven Myths of Human Annotation” AI Magazine 36.1: 15-24 (2015).
[6] Hovy et al. “Learning Whom

In [10]:
from llama_index.llms.ollama import Ollama
Settings.llm = Ollama(model="granite4:3b", request_timeout=200)

query_engine = index.as_query_engine(
    similarity_top_k=3
)

response = query_engine.query(
    "Is There an articel by Francis Galton published in Nature?"
)
print(response)

Yes, there is an article by Francis Galton published in Nature. In 1907, he published a paper titled "Vox populi" in Nature magazine.


# 6. Ask the model direct

In [12]:
import ollama

client = ollama.Client()

response = client.chat(
    model="granite4:3b",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a factual assistant. "
                "Answer ONLY using verifiable information from the given source. "
                "If you are unsure or the information is not available, say "
                "'I don't know based on the available information.' "
                "Do NOT speculate, infer, or fabricate facts."
            )
        },
        {
            "role": "user",
            "content": (
                "Is There an article by Francis Galton published in Nature?"
            )
        }
    ],
    options={
        "temperature": 0.0
    }
)

print(response["message"]["content"])


I don't know based on the available information. The source provided does not contain any information about an article by Francis Galton published in Nature.


In [14]:
###### look into the sqlite db #######
import sqlite3

conn = sqlite3.connect("./chroma/chroma.sqlite3")
cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name;
""")

tables = cursor.fetchall()
print("Tables:", tables)

conn.close()


Tables: [('acquire_write',), ('collection_metadata',), ('collections',), ('databases',), ('embedding_fulltext_search',), ('embedding_fulltext_search_config',), ('embedding_fulltext_search_content',), ('embedding_fulltext_search_data',), ('embedding_fulltext_search_docsize',), ('embedding_fulltext_search_idx',), ('embedding_metadata',), ('embeddings',), ('embeddings_queue',), ('embeddings_queue_config',), ('maintenance_log',), ('max_seq_id',), ('migrations',), ('segment_metadata',), ('segments',), ('tenants',)]


In [16]:
print(sqlite3.connect("./chroma/chroma.sqlite3").execute("SELECT * FROM collections").fetchall())

[('fb53302e-f426-48a2-abc4-c9e9ac6f6f26', 'lilianweng', 384, '00000000-0000-0000-0000-000000000000', '{}', '{"defaults":{"string":{"fts_index":{"enabled":false,"config":{}},"string_inverted_index":{"enabled":true,"config":{}}},"float_list":{"vector_index":{"enabled":false,"config":{"space":"l2","embedding_function":{"type":"known","name":"default","config":{}},"hnsw":{"ef_construction":100,"max_neighbors":16,"ef_search":100,"num_threads":12,"batch_size":100,"sync_threshold":1000,"resize_factor":1.2}}}},"sparse_vector":{"sparse_vector_index":{"enabled":false,"config":{"embedding_function":{"type":"unknown"},"bm25":false}}},"int":{"int_inverted_index":{"enabled":true,"config":{}}},"float":{"float_inverted_index":{"enabled":true,"config":{}}},"bool":{"bool_inverted_index":{"enabled":true,"config":{}}}},"keys":{"URL":{"string":{"fts_index":{"enabled":false,"config":{}},"string_inverted_index":{"enabled":true,"config":{}}}},"doc_id":{"string":{"fts_index":{"enabled":false,"config":{}},"stri